# 06. Batch, Torch, 그리고 다음 병목

**목표**
- 관점을 단일 solve latency에서 batch throughput과 differentiable solve로 옮긴다.
- 같은 topology에서 여러 `Sbus` scenario를 푸는 문제 설정을 확인한다.
- Torch/autograd interface가 필요한 이유와 현재 환경의 한계를 구분한다.

**대표 응용**
- contingency analysis
- time-series simulation
- Monte Carlo sampling
- optimization 또는 learning loop 안의 differentiable power flow

**핵심 변화**
- 01~05: 한 operating point를 얼마나 빠르게 푸는가
- 06: 많은 operating point와 gradient를 어떻게 다룰 것인가


In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'cuPF').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from python.tutorial import tutorial_utils as tu

plt.rcParams['figure.figsize'] = (8, 4.8)
plt.rcParams['axes.grid'] = False
pd.set_option('display.max_colwidth', 120)

RUN_EXAMPLES = True


**`solve_batch`가 표현하는 문제**
- 같은 `Ybus`, 같은 PV/PQ 구조를 재사용한다.
- 여러 `Sbus`, `V0`를 batch-major 배열로 넘긴다.
- CPU path는 batch size 1 중심이다.
- batch > 1은 CUDA FP32/Mixed path에서 더 의미가 있다.


In [2]:
try:
    import torch
    has_cuda = torch.cuda.is_available()
except Exception:
    has_cuda = False

case = tu.load_case('case9')
cupf = None
kind = 'gpu' if has_cuda else 'cpu'
if has_cuda:
    build = tu.build_eval('gpu', jobs=2, timeout=3600)
    print(tu.command_summary(build, tail_lines=14))
    cupf, msg = tu.import_cupf_from_build('gpu')
else:
    build = tu.build_eval('cpu', jobs=2, timeout=2400)
    print(tu.command_summary(build, tail_lines=14))
    cupf, msg = tu.import_cupf_from_build('cpu')
print(msg)


$ bash benchmark/scripts/build_eval.bash gpu --jobs 2
[OK] elapsed=0.3s
-- Configuring done
-- Generating done
-- Build files have been written to: /workspace/gpu-powerflow-master/cuPF/build/eval-gpu
[build_eval] build _cupf + cupf_cpp_evaluate (-j 2)
Consolidate compiler generated dependencies of target cupf
[ 89%] Built target cupf
Consolidate compiler generated dependencies of target _cupf
[100%] Built target _cupf
[ 86%] Built target cupf
Consolidate compiler generated dependencies of target cupf_cpp_evaluate
[100%] Built target cupf_cpp_evaluate
[build_eval] done. artifacts under /workspace/gpu-powerflow-master/cuPF/build/eval-gpu
[build_eval]   /workspace/gpu-powerflow-master/cuPF/build/eval-gpu/tests/cupf_cpp_evaluate
[build_eval]   /workspace/gpu-powerflow-master/cuPF/build/eval-gpu/_cupf.cpython-310-x86_64-linux-gnu.so
imported from eval-gpu


In [3]:
batch_result_table = pd.DataFrame()
if cupf is None:
    print('cuPF Python module is unavailable, so the batch example cannot run.')
else:
    try:
        opts = cupf.NewtonOptions()
        if kind == 'gpu':
            opts.backend = cupf.BackendKind.CUDA
            opts.compute = cupf.ComputePolicy.Mixed
            scales = np.array([1.00, 1.01, 0.99, 1.02])
        else:
            opts.backend = cupf.BackendKind.CPU
            opts.compute = cupf.ComputePolicy.FP64
            opts.cpu_linear_solver = cupf.CpuLinearSolverKind.KLU
            scales = np.array([1.00])
        solver = cupf.NewtonSolver(opts)
        solver.initialize(case.ybus.indptr, case.ybus.indices, case.ybus.data, case.ybus.shape[0], case.ybus.shape[1], case.pv, case.pq)
        sbus_batch = np.stack([case.sbus * scale for scale in scales]).astype(np.complex128)
        v0_batch = np.stack([case.v0 for _ in scales]).astype(np.complex128)
        config = cupf.NRConfig()
        config.tolerance = 1e-6 if kind == 'gpu' else 1e-8
        config.max_iter = 50
        result = solver.solve_batch(
            case.ybus.indptr, case.ybus.indices, case.ybus.data,
            case.ybus.shape[0], case.ybus.shape[1],
            sbus_batch, v0_batch, case.pv, case.pq, config,
        )
        batch_result_table = pd.DataFrame({
            'scenario': np.arange(result.batch_size),
            'load_scale': scales,
            'backend': kind,
            'tolerance': config.tolerance,
            'converged': result.converged_numpy.astype(bool),
            'iterations': result.iterations_numpy,
            'final_mismatch': result.final_mismatch_numpy,
        })
        display(batch_result_table)
    except Exception as exc:
        print(f'batch solve unavailable in this configuration: {type(exc).__name__}: {exc}')


,scenario,load_scale,backend,tolerance,converged,iterations,final_mismatch
0,0,1.00,gpu,0.000001,True,4,3.424897e-07
1,1,1.01,gpu,0.000001,True,4,3.515191e-07
2,2,0.99,gpu,0.000001,True,4,3.323748e-07
3,3,1.02,gpu,0.000001,True,4,3.641224e-07


In [4]:
if not batch_result_table.empty:
    worst = float(batch_result_table['final_mismatch'].max())
    tol = float(batch_result_table['tolerance'].iloc[0])
    all_ok = bool(batch_result_table['converged'].all() and worst <= tol)
    text = f"- batch 예제는 같은 topology에서 `Sbus`만 바꾼 scenario를 한 번에 넘기는 모양을 보여준다. worst mismatch={worst:.3e}, tolerance={tol:.1e}, all_ok={all_ok}."
    if kind == 'gpu':
        text += " Mixed precision path라서 tolerance를 `1e-6`으로 두었다. 이 숫자는 02~05의 FP64 `1e-8` 결과와 같은 정확도 주장으로 읽으면 안 된다."
    display(Markdown(text))
else:
    display(Markdown('- batch 결과가 없으면 이 절은 API 모양만 확인한 것이다. CUDA/pybind build 상태를 먼저 확인한다.'))


- batch 예제는 같은 topology에서 `Sbus`만 바꾼 scenario를 한 번에 넘기는 모양을 보여준다. worst mismatch=3.641e-07, tolerance=1.0e-06, all_ok=True. Mixed precision path라서 tolerance를 `1e-6`으로 두었다. 이 숫자는 02~05의 FP64 `1e-8` 결과와 같은 정확도 주장으로 읽으면 안 된다.

**Torch autograd가 답하는 질문**
- 질문: load perturbation이 voltage와 downstream loss에 어떤 영향을 주는가
- interface: forward Newton solve와 adjoint solve를 연결한다.
- 주의: Torch extension이 없는 환경에서는 gradient 결과를 주장하지 않는다.


In [5]:
torch_status = {'status': 'not_run', 'reason': ''}
try:
    import torch
    if not torch.cuda.is_available():
        torch_status = {'status': 'skipped', 'reason': 'Torch CUDA is unavailable'}
        print('Torch autograd example needs CUDA.')
    else:
        cupf_gpu, _ = tu.import_cupf_from_build('gpu')
        if cupf_gpu is None or getattr(cupf_gpu, 'solve', None) is None or not hasattr(cupf_gpu.NewtonSolver, 'solve_with_adjoint_cache_torch'):
            torch_status = {'status': 'skipped', 'reason': 'cuPF was built without Torch extension methods'}
            print('cuPF was built without the Torch extension methods in this environment.')
        else:
            opts = cupf_gpu.NewtonOptions()
            opts.backend = cupf_gpu.BackendKind.CUDA
            opts.compute = cupf_gpu.ComputePolicy.Mixed
            solver = cupf_gpu.NewtonSolver(opts)
            solver.initialize(case.ybus.indptr, case.ybus.indices, case.ybus.data, case.ybus.shape[0], case.ybus.shape[1], case.pv, case.pq)
            device = torch.device('cuda')
            dtype = torch.float32
            load_p = torch.zeros((2, case.ybus.shape[0]), device=device, dtype=dtype, requires_grad=True)
            load_q = torch.zeros((2, case.ybus.shape[0]), device=device, dtype=dtype, requires_grad=True)
            load_p.data[1] = 0.01
            sbus_base_re = torch.tensor(case.sbus.real, device=device, dtype=dtype)
            sbus_base_im = torch.tensor(case.sbus.imag, device=device, dtype=dtype)
            v0_va = torch.angle(torch.tensor(case.v0, device=device, dtype=torch.complex64)).to(dtype)
            v0_vm = torch.abs(torch.tensor(case.v0, device=device, dtype=torch.complex64)).to(dtype)
            config = cupf_gpu.NRConfig(); config.tolerance = 1e-6; config.max_iter = 30
            va, vm = cupf_gpu.solve(load_p, load_q, solver, sbus_base_re=sbus_base_re, sbus_base_im=sbus_base_im, v0_va=v0_va, v0_vm=v0_vm, config=config, solve_options=cupf_gpu.SolveOptions())
            loss = va[:, 1].sum() + vm[:, 1].sum()
            loss.backward()
            torch_status = {'status': 'ok', 'reason': ''}
            display(pd.DataFrame([{'va_shape': tuple(va.shape), 'vm_shape': tuple(vm.shape), 'grad_p_finite': bool(torch.isfinite(load_p.grad).all()), 'grad_q_finite': bool(torch.isfinite(load_q.grad).all())}]))
except Exception as exc:
    torch_status = {'status': 'failed', 'reason': f'{type(exc).__name__}: {exc}'}
    print(f'Torch autograd example failed: {type(exc).__name__}: {exc}')


cuPF was built without the Torch extension methods in this environment.


In [6]:
display(pd.DataFrame([torch_status]))
if torch_status['status'] == 'ok':
    display(Markdown('- Torch 절은 실제 gradient까지 계산했다. 여기서 gradient는 load perturbation이 선택한 voltage-derived loss에 주는 sensitivity다.'))
else:
    display(Markdown(f"- Torch 절은 이 환경에서 gradient 결과를 주장하지 않는다. 상태는 `{torch_status['status']}`이고 이유는 `{torch_status['reason']}`이다. 이 경우 이 절의 의미는 필요한 extension/API 조건을 명확히 하는 데 있다."))


,status,reason
0,skipped,cuPF was built without Torch extension methods


- Torch 절은 이 환경에서 gradient 결과를 주장하지 않는다. 상태는 `skipped`이고 이유는 `cuPF was built without Torch extension methods`이다. 이 경우 이 절의 의미는 필요한 extension/API 조건을 명확히 하는 데 있다.

**남은 연구 방향**

| 방향 | 앞에서 본 병목과의 연결 |
|---|---|
| custom linear solver | cuDSS가 지배적인 비용이면 전력계통 Jacobian 구조를 더 이용해야 한다. 05에서 custom path가 빨라지면 이 가설이 강해지고, 아니면 병목을 더 세분화해야 한다. |
| cuGraph / graph scheduling | Edge/Vertex Jacobian fill은 graph traversal와 sparse scatter/gather 문제다. 05에서 edge/vertex 차이가 작으면 전체 solve가 fill보다 factorization에 지배된다는 해석과 연결된다. |
| multi-GPU | 단일 solve latency보다 scenario batch 처리에서 자연스럽다. 06의 batch API가 이 문제 설정을 만든다. |
| tensor core / mixed precision | FP64 비용을 줄일 수 있지만 06의 mixed tolerance처럼 정확도 기준을 함께 말해야 한다. |
| iterative refinement | mixed precision solve를 Newton tolerance와 연결하는 안전장치다. |

**전체 흐름 요약**
- 전력망은 `Ybus`와 `Sbus`로 바뀐다.
- Newton은 mismatch와 Jacobian을 반복해서 만든다.
- baseline은 이 구조를 신뢰할 수 있는 기준으로 제공한다.
- cuPF는 sparse pattern, solver backend, GPU batch라는 반복성을 더 강하게 이용하려는 시도다.
- 남은 연구는 더 많은 graph와 scenario에서 이 반복성을 얼마나 안정적으로 끌어내느냐의 문제다.
